# Phase 2 — Triplet Fine-Tuning, Image+Text Fusion, FAISS/ANNOY

**Run Phase 1 first** (`notebooks/phase1_baseline.ipynb`) — this notebook reuses
its data splits (`data/splits/*.csv`) and expects them to already exist.
Do not treat any numbers here as final until you've compared them against
your approved Phase 1 baseline under the same test set and Protocol A.

This notebook is a thin wrapper: all logic lives in `src/` and `scripts/`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = "/content/drive/MyDrive/fashion_recommender_project"  # <-- adjust to your path
%cd {PROJECT_DIR}


In [ ]:
!pip install -q -r requirements.txt


## 1. Load Phase 1 splits (must already exist)

In [ ]:
import sys
sys.path.append(".")
from src import config, preprocess

config.ensure_directories()
train_df, val_df, test_df = preprocess.load_splits()
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")


## 2. Triplet-loss fine-tuning
Fine-tunes only the last `config.UNFREEZE_LAST_N_BLOCKS` ResNet50 block(s)
(layer4 by default) with `nn.TripletMarginLoss`, using randomly-sampled
anchor/positive/negative triplets keyed on `articleType`
(see `src/triplet_sampling.py`). Hyperparameters are in
`configs/phase2_config.yaml`.


In [ ]:
from src import finetune

model, training_log_df = finetune.finetune_with_triplet_loss(train_df)
training_log_df


## 3. Extract fine-tuned embeddings for all splits

In [ ]:
from src import data_loader, embeddings

for split_name, df in [("train", train_df), ("val", val_df), ("test", test_df)]:
    loader = data_loader.build_dataloader(df, shuffle=False)
    emb, ids, cats, elapsed, n_images = embeddings.extract_embeddings(model, loader, split_name)
    embeddings.save_embeddings(split_name, emb, ids, cats, tag="finetuned")


## 4. Fine-tuned exact k-NN evaluation (same protocol as Phase 1)
Compare these numbers directly against `results/metrics/metrics_baseline_exact_knn.csv`
from Phase 1 — same test set, same Protocol A, same K values. This answers
RQ2 (does triplet-loss fine-tuning improve retrieval over the pretrained baseline?).


In [ ]:
from src import evaluation, retrieval

emb_train, ids_train, cat_train = embeddings.load_embeddings("train", tag="finetuned")
emb_test, ids_test, cat_test = embeddings.load_embeddings("test", tag="finetuned")

knn_index, _ = retrieval.build_knn_index(emb_train, k_max=config.K_MAX)
neighbor_ids, neighbor_sims, _ = retrieval.retrieve_top_k(emb_test, ids_test, ids_train, k=config.K_MAX, index=knn_index)

gallery_lookup = evaluation.build_category_lookup(ids_train, cat_train)
pr_results = evaluation.precision_recall_at_k(neighbor_ids, gallery_lookup, cat_train, cat_test, config.K_VALUES)
map_results = {k: evaluation.mean_average_precision_at_k(neighbor_ids, gallery_lookup, cat_test, k) for k in config.MAP_K_VALUES}
cat_consistency = {k: evaluation.category_consistency_at_k(neighbor_ids, gallery_lookup, cat_test, k) for k in config.CATEGORY_CONSISTENCY_K_VALUES}

for k in config.K_VALUES:
    print(f"Precision@{k}: {pr_results[k]['precision']:.4f}  Recall@{k}: {pr_results[k]['recall']:.4f}")
for k, v in map_results.items():
    print(f"mAP@{k}: {v:.4f}")

finetuned_metrics_df = evaluation.metrics_to_dataframe(pr_results, map_results, cat_consistency)
finetuned_metrics_df.to_csv(config.METRICS_DIR / "metrics_finetuned_exact_knn.csv", index=False)
finetuned_metrics_df


## 5. Image + text fusion (RQ3)
Fuses the fine-tuned image embeddings with product-title text embeddings
(TF-IDF by default; SBERT optional via `configs/phase2_config.yaml`) and
compares against image-only retrieval under the same protocol.


In [ ]:
!python scripts/evaluate_phase2.py


In [ ]:
import pandas as pd
pd.read_csv(config.METRICS_DIR / "phase2_image_only_vs_fused.csv")


## 6. FAISS/ANNOY approximate search vs. exact k-NN (RQ4)
Benchmarks the configured ANN backend (`configs/phase2_config.yaml` ->
`ann_backend`) against exact k-NN on the fine-tuned embeddings: query
latency, Precision@K, and Top-K overlap with the exact results.


In [ ]:
!python scripts/build_ann_index.py --tag finetuned --k 10


In [ ]:
pd.read_csv(config.METRICS_DIR / f"exact_vs_{config.ANN_BACKEND}_finetuned.csv")


## Phase 2 stop gate
Check `results/metrics/` for:
- `metrics_finetuned_exact_knn.csv` — fine-tuned vs. Phase 1 baseline (RQ2)
- `phase2_image_only_vs_fused.csv` — image-only vs. fused (RQ3)
- `exact_vs_{ann_backend}_finetuned.csv` — exact vs. approximate search (RQ4)

And `results/figures/` for the corresponding plots. Compile these into the
final paper-results section once you're satisfied every comparison used the
same test set and Protocol A throughout.
